# 项目架构与进展分析笔记

## 动机

用户要求重述当前项目做了哪些事情。我们需要梳理项目结构，以明确现有代码实现了哪些对比方法（baselines）和 `vcaan` 模型，从而为将来开发新改进的模型（Improved Model）做好准备。

## 1. 现有项目总结

从项目结构和代码来看，这是一个专门为了**多变量时间序列缺失值插补 (Multivariate Time Series Imputation)** 搭建的标准化实验框架，所有的实验都可以通过 `main.py` 流水线式运行。

目前已经完成并可用的工作包括：
1. **数据集准备流水线**（`src/data`）：支持构建不同缺失模式（`mcar`, `seq`连续缺失, `scm`空间相关的缺失）和通过指定的缺失率（10%, 30%, 50%）自动生成对应的掩码(Mask)。支持时间序列按年按月进行固定的 Train/Val/Test 划分。最后也实现了特征的时间窗切分以送入深度模型。
2. **完整的基线对比方法库**（`src/models/baselines.py`）：
   - **PyPOTS（深度学习类）**：包含了 `saits`（基于 Self-Attention）, `grud`（基于 RNN）, `usgan`（基于 GAN）, `itransformer`。
   - **机器学习方法**：包含了 `knn`, `mice`（代码中还实现了基于分块 `Chunked` 的方法以避免长序列预测过慢的问题）。
   - **简易统计方法**：如 `locf`（最近观测值结转）。
3. **前期论文模型：VCAAN**（`src/models/vcaan.py`）：用户提到的前期已发表论文的模型。在当前代码中，实现为一种迭代优化的时序空间模型：它**先使用 LOCF 进行预填充（Warm Start）**，接着提取时空相关性特征矩阵，然后利用**线性回归迭代式**优化缺失位置的取值，直至达到收敛阈值。它作为一个baseline选项集成在了这个框架中。
4. **标准的评估与自动化日志系统**：
   - **评价指标（`src/evaluate.py`）**：严格计算了只有真实**缺失位置（missing positions）上的 RMSE**。
   - **日志与可视化**：每次实验记录详细 config 与指标表到 `logs/` 目录；`scripts/plot_baseline_compare.py` 支持将不同模型的结果绘制为对比柱状图。

## 2. 下一步改进动机

项目的脚手架（数据划分、缺失构造、训练测试流程、统一评测标准）已经非常健康且标准化。用户的动机是：**如果想要在这个标准化跑道上，实现一个新的、改进的插补模型**。这就意味着我们在接下来的步骤中，架构上只需要：
1. 在 `src/models/` 目录下构思并新建新模型的核心代码逻辑。
2. 将新模型注册和桥接到 `src/models/baselines.py` 和 `main.py` 的解析参数中。
3. 新模型可直接通过 `main.py` 复用已有的数据集和评价体系，无缝地和 VCAAN 及其他已有基线同台比对了。

### 🚀 优化日志输出 (Reduced Verbosity)
1. **分析**: 实验跑批脚本 `run_experiments.sh` 在调用 `main.py` 时默认没有传入 `--quiet-train`，导致深度学习模型（如 saits, grud 等）或者其他迭代算法输出大量 epoch 级别的日志或进度条，使得日志文件极大且难以抓住关键信息。在查阅 `src/models/pypots_baselines.py` 等代码时我们发现，它内部的确是根据 `verbose` 参数将状态配置给了模型。
2. **操作**:
   - 在 `run_experiments.sh` 参数体系中，追加了 `--quiet-train` 功能以静默每轮迭代的重复训练日志。
   - **核心保留**: 由 `runner.py` 保留了参数配置、环境信息、数据切分信息、每个模型的最佳 HPO 和最终评测指标（RMSE和时间）。


### 🚀 进一步精简日志: 隐藏重复保存提示
1. **分析**: 之前发现在跑批时，由于 `src/experiments/runner.py` 采取了增量保存的设计（每个组合跑完后立刻调用 `save_experiment_results`），这导致 `src/experiments/results.py` 末尾的 `print` 语句会被频繁触发。每一轮都会完整打印一份越来越大的 Pivot 结果表和4条文件保存的路径信息，产生了大量刷屏垃圾。
2. **操作**:
   - 移除了 `src/experiments/results.py` 中  最后的批量 `print` 语句。
   - 保留了 `runner.py` 里对于 **当前运行轮次** 的核心指标打印（如  和 ）。这样既能实时监控进度，又不会被滚雪球般的历史表格淹没。


### 🚀 进一步精简日志: 隐藏重复保存提示
1. **分析**: 之前发现在跑批时，由于 `src/experiments/runner.py` 采取了增量保存的设计（每个组合跑完后立刻调用 `save_experiment_results`），这导致 `src/experiments/results.py` 末尾的 `print` 语句会被频繁触发。每一轮都会完整打印一份越来越大的 Pivot 结果表和4条文件保存的路径信息，产生了大量刷屏垃圾。
2. **操作**:
   - 移除了 `src/experiments/results.py` 中 save_experiment_results 最后的批量 `print` 语句。
   - 保留了 `runner.py` 里对于 **当前运行轮次** 的核心指标打印（如 RMSE train/val/test 和 Timing）。这样既能实时监控进度，又不会被滚雪球般的历史表格淹没。


### 🚀 引入 `rich` 结构化终端输出
1. **分析**: 用户希望在保留精简日志的基础上，让输出看起来像命令行结构化界面一样直观、美观。
2. **操作**:
   - 引入了 `rich` 库 (`uv add rich`)。
   - 在 `src/experiments/runner.py` 中初始化了 `rich.table.Table`，并使用 `rich.live.Live` 进行动态渲染。
   - 每次模型+模式+缺失率探索完后，动态向表格追加一行结果 (包含 `Model`, `Pattern`, `PI`, `Val RMSE`, `Test RMSE`, `Time`)。
   - 额外地，通过配置 Python `logging`，从根源静默了 `pypots` 未提供 `saving_path` 时的 `WARNING`。


### 🚀 终极日志清理: 冻结 PyPOTS 内置 Logger
1. **分析**: 尽管我们在外部设置了 `logger_creator.set_level('error')`， 但进入模型时，跑批脚本传递了 `verbose=False`，导致 `PyPOTS.BaseModel` 内部初始化阶段强行执行了 `logger_creator.set_level('warning')`，将我们的 `ERROR` 级别覆盖并打印了烦人的 `saving_path not given` 警告。
2. **操作**:
   - 在 `src/experiments/runner.py` 中，不仅设置了 `logger_creator.set_level('error')`，还利用了 Python 动态语言特性对内置日志器实施**降维打击**: `pypots_logger_creator.set_level = lambda *args, **kwargs: None`。
   - 这样一来，在整个生命周期内，PyPOTS 的 `WARNING` 以及运行设备检查的 `INFO` 就被彻底“噤声”了。
   - 现在的实验控制台输出极为纯净：只剩下开局参数信息和跑批时的结构化进度表格！


### 🌈 进度监控及日志排版升级
1. **参数更易读化**: 将原本单行、杂乱平铺的 `args` 字典转化成了结构化的 `JSON` 格式输出 (`json.dumps(..., indent=2)`)，使得在每次实验开始时能够非常直观地检视所有生效的配置。
2. **实时模型训练指示器**: 在静默了 PyPOTS （即通过 `--quiet-train`）的大量输出后，每次针对一套“模型+缺失模式+缺失率”组合进行训练时，控制台显得仿佛停滞了。
    - 针对此问题，我们基于 `rich.console.Group` 与 `rich.text.Text`，在原本的实时结果表格 () 上方，追加了一个高亮的动态进度条：`⏳ Currently running: Model=saits | Pattern=mcar | PI=0.10`。
    - 只要进入下一层循环，文字就会自动变为正在计算的参数；而所有组合计算结束后，我们强制执行 `live.update(result_table)`，将最后一轮的临时动态字样彻底抹除，只留下一张完美纯净的汇总表格。


### 🌈 进度监控及日志排版升级
1. **参数更易读化**: 将原本单行、杂乱平铺的 `args` 字典转化成了结构化的 `JSON` 格式输出 (`json.dumps(..., indent=2)`)，使得在每次实验开始时能够非常直观地检视所有生效的配置。
2. **实时模型训练指示器**: 在静默了 PyPOTS （即通过 `--quiet-train`）的大量输出后，每次针对一套“模型+缺失模式+缺失率”组合进行训练时，控制台显得仿佛停滞了。
    - 针对此问题，我们基于 `rich.console.Group` 与 `rich.text.Text`，在原本的实时结果表格 (`result_table`) 上方，追加了一个高亮的动态进度条：`⏳ Currently running: Model=saits | Pattern=mcar | PI=0.10`。
    - 只要进入下一层循环，文字就会自动变为正在计算的参数；而所有组合计算结束后，我们强制执行 `live.update(result_table)`，将最后一轮的临时动态字样彻底抹除，只留下一张完美纯净的汇总表格。


### 🐛 深入修复: 进度条在 Shell 脚本中不渲染的问题
在使用 `./run_experiments.sh` 时，我发现之前利用 `rich` 添加的实时进度条 (Live & Group) 虽然在直接敲击 Python 命令时有效，但在被 Shell 跑批脚本调用时却“隐身”了。

1. **寻根溯源**:
   主要原因是我们在 `runner.py` 里运用了双路日志分发技巧(`Tee`)，把所有的标准输出拦截并写入了 `train.log` 中。而 `rich.Console()` 默认绑定到了当前的 `sys.stdout` (此时已被 `Tee` 劫持)，这就导致 rich 库底层的控制台终端检测机制受挫，直接丢弃或无法输出带有 ANSI 动画转义码的字符串。

2. **解决手段**:
   必须让 `rich` 渲染层“越过”我们的 `Tee` 拦截网。于是我在初始化阶段强制告诉 `rich`：“你要直接跟最底层的真实屏幕流对话！” 
   ```python
   import sys as _sys
   console = Console(file=_sys.__stdout__)
   ```
   **通过直接绑定 `sys.__stdout__`**，我们完美拆分了输出流：
   - 所有的单纯 `print` 或外部模型的普通日志依然会被 `Tee` 捕捉并忠实地记录进 `train.log` 中。
   - 而这根炫酷且实时的 `rich` 进度条及结果表格，则独享一条 V.I.P. 通道直接在终端闪烁刷新，且不会用脏乱的 ANSI 码污染纯文本的日志文件！
